# PerDev fine-tune: teach Qwen2.5-Coder-7B to be PerDev's agent

Trains a LoRA adapter on real, test-verified bug-fix sessions from
[SWE-smith trajectories](https://huggingface.co/datasets/SWE-bench/SWE-smith-trajectories) (MIT license),
rewritten into PerDev's exact tools and system prompt. You can add your own PerDev logs as well.

**How to run on Kaggle (free):**
1. *Settings → Accelerator → GPU T4 x2* and *Settings → Internet → On* (your account needs a verified phone number for GPUs).
2. Optional: upload your `chat.jsonl` from `node scripts/export-training-data.js data` as a Kaggle dataset and add it to this notebook.
3. Click **Save Version → Save & Run All (Commit)**. It runs in the background (up to 12 h; this stops itself at `MAX_HOURS`) and you can close the tab.
4. When it finishes, open the version's **Output** tab and download `perdev-lora-f16.gguf` (about 80 MB).

The last cell prints a before/after score so you can see whether the adapter helps.

In [ ]:
%%capture
%pip install unsloth

In [ ]:
%%writefile perdev_format.json
{
  "tools": [
    {
      "type": "function",
      "function": {
        "name": "read_file",
        "description": "Read a text file. Returns lines prefixed with line numbers. Use offset/limit for big files.",
        "parameters": {
          "type": "object",
          "properties": {
            "path": {
              "type": "string",
              "description": "Path relative to the workspace root"
            },
            "offset": {
              "type": "number",
              "description": "1-based line to start at (optional)"
            },
            "limit": {
              "type": "number",
              "description": "Number of lines to read (optional, default 400)"
            }
          },
          "required": [
            "path"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "write_file",
        "description": "Create a new file or completely overwrite an existing one. To change part of an existing file, use edit_file instead.",
        "parameters": {
          "type": "object",
          "properties": {
            "path": {
              "type": "string",
              "description": "Path relative to the workspace root"
            },
            "content": {
              "type": "string",
              "description": "Full file content, exactly as it should be saved (no line numbers)"
            }
          },
          "required": [
            "path",
            "content"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "edit_file",
        "description": "Replace an exact snippet in a file. This is the preferred way to change existing files. old_str must match the file exactly (including indentation) and occur exactly once. Read the file first.",
        "parameters": {
          "type": "object",
          "properties": {
            "path": {
              "type": "string"
            },
            "old_str": {
              "type": "string",
              "description": "Exact text to replace, copied from the file without the line numbers read_file shows"
            },
            "new_str": {
              "type": "string",
              "description": "Replacement text (no line numbers)"
            }
          },
          "required": [
            "path",
            "old_str",
            "new_str"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "list_files",
        "description": "List files in the workspace matching a glob pattern, e.g. \"src/**/*.ts\". Defaults to all files.",
        "parameters": {
          "type": "object",
          "properties": {
            "pattern": {
              "type": "string",
              "description": "Glob pattern (optional)"
            }
          }
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "search",
        "description": "Search file contents with a regular expression. Returns matching lines as path:line: text.",
        "parameters": {
          "type": "object",
          "properties": {
            "regex": {
              "type": "string",
              "description": "JavaScript regular expression"
            },
            "glob": {
              "type": "string",
              "description": "Limit to files matching this glob (optional)"
            }
          },
          "required": [
            "regex"
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "run_command",
        "description": "Run a shell command in the workspace root (e.g. tests, builds, git status). Returns exit code and output. Times out after 120s.",
        "parameters": {
          "type": "object",
          "properties": {
            "command": {
              "type": "string"
            }
          },
          "required": [
            "command"
          ]
        }
      }
    }
  ],
  "max_output": 12000,
  "no_output_note": "(the command finished and printed nothing)",
  "system_prompt_template": "You are PerDev, an expert software engineer working inside the user's code editor.\n\nWorkspace root: {root}\nOS: {platform} ({release})\nShell used by run_command: {shell}\nDate: {date}\n\nYou can call tools to read, search, create and edit files and to run shell commands. Call them yourself; never ask the user to run a tool for you.\nHow to work:\n- Explore before changing things: use list_files, search and read_file to understand the code.\n- Make focused changes with edit_file. Always read a file before editing it, and copy old_str exactly.\n- Use write_file only for new files or full rewrites.\n- After changing code, verify it: run the tests, the build or the program with run_command when possible, and fix what fails.\n- A command that prints nothing has still run. Don't repeat a call whose result you already have; use the result.\n- Paths are relative to the workspace root.\n- When the task is complete, reply with a short summary of what you changed. Do not call tools in that final reply.\n- If the request is just a question, answer it directly and concisely, reading files first if you need to."
}

In [ ]:
%%writefile build_dataset.py
"""Convert free, permissively licensed agent trajectories into PerDev's format.

Source: SWE-bench/SWE-smith-trajectories (MIT), "tool" split: real bug-fix
sessions with tool calls, each verified by the repo's tests ("resolved").

Each trajectory is rewritten so the model learns PerDev's exact interface:
  bash                            -> run_command
  str_replace_editor view (file)  -> read_file   (output in PerDev's line format)
  str_replace_editor view (dir)   -> list_files
  str_replace_editor create       -> write_file
  str_replace_editor str_replace  -> edit_file
  submit                          -> final answer (no tool call)
with PerDev's system prompt, workspace-relative paths and the same output
truncation the extension applies. Trajectories that can't be mapped cleanly
(insert/undo_edit, paths outside the repo, too long) are skipped.

Standard library only. Usage:
  python build_dataset.py --input rows.json --out train.jsonl
"""

from __future__ import annotations

import argparse
import json
import re
from pathlib import Path

HERE = Path(__file__).resolve().parent
FORMAT = json.loads((HERE / "perdev_format.json").read_text(encoding="utf-8"))
MAX_OUTPUT = FORMAT["max_output"]
SRC_ROOT = "/testbed"


class Skip(Exception):
    """Trajectory can't be converted faithfully."""


def truncate(s: str) -> str:
    # Mirrors truncate() in src/toolSpecs.ts.
    if len(s) <= MAX_OUTPUT:
        return s
    half = MAX_OUTPUT // 2
    return f"{s[:half]}\n\n... [{len(s) - MAX_OUTPUT} characters truncated] ...\n\n{s[-half:]}"


def system_prompt(repo: str) -> str:
    return (
        FORMAT["system_prompt_template"]
        .replace("{root}", f"/home/dev/{repo}")
        .replace("{platform}", "linux")
        .replace("{release}", "6.8.0")
        .replace("{date}", "2025-06-01")
        .replace("{shell}", "bash")
    )


def rel_path(p: str) -> str:
    p = p.strip()
    if p in (SRC_ROOT, SRC_ROOT + "/"):
        return "."
    if p.startswith(SRC_ROOT + "/"):
        return p[len(SRC_ROOT) + 1 :]
    if p.startswith("/"):
        raise Skip(f"path outside repo: {p}")
    return p


def rel_text(s: str) -> str:
    """Make /testbed paths in commands and outputs workspace-relative."""
    s = re.sub(r"cd /testbed\s*(&&|;)\s*", "", s)
    s = s.replace(SRC_ROOT + "/", "")
    return re.sub(r"/testbed\b", ".", s)


def text_of(content) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    return "".join(part.get("text", "") for part in content if isinstance(part, dict))


def clean_user(text: str) -> str:
    text = re.sub(r"<uploaded_files>\s*/testbed\s*</uploaded_files>\s*", "", text)
    text = text.replace("in the directory /testbed", "in the current workspace")
    text = re.sub(r"\b(in|to|of) /testbed\b", r"\1 the workspace", text)
    return rel_text(text)


def observation(raw: str) -> str:
    raw = text_of(raw)
    return re.sub(r"^OBSERVATION:\s*\n?", "", raw)


def convert_view(args: dict, obs: str):
    path = rel_path(args["path"])
    if obs.startswith("Here's the files and directories"):
        lines = [rel_text(l.strip()) for l in obs.splitlines()[1:] if l.strip()]
        pattern = "**/*" if path == "." else f"{path}/**/*"
        return "list_files", {"pattern": pattern}, "\n".join(lines) or "No files matched."
    abbreviated = obs.startswith("<NOTE>This file is too large")
    if not abbreviated and (not obs or "cat -n" not in obs.splitlines()[0]):
        raise Skip("unexpected view output")
    out_lines = []
    for line in obs.splitlines()[1:]:
        m = re.match(r"^\s*(\d+)\t(.*)$", line)
        if m:
            out_lines.append(f"{int(m.group(1)):>5}  {m.group(2)}")
        elif abbreviated and line.strip().startswith("..."):
            out_lines.append("  ...")
    if abbreviated:
        # SWE-agent shows an outline of big files; keep it, with PerDev-style guidance.
        out_lines.append("[Large file: only an outline is shown; use offset and limit to read specific lines]")
    new_args = {"path": path}
    rng = args.get("view_range")
    if rng and len(rng) == 2:
        start, end = int(rng[0]), int(rng[1])
        new_args["offset"] = start
        if end != -1:
            new_args["limit"] = end - start + 1
    return "read_file", new_args, rel_text("\n".join(out_lines))


def convert_editor(args: dict, obs: str):
    cmd = args.get("command")
    path = rel_path(args.get("path", ""))
    if re.match(r"The path \S+ does not exist", obs) and cmd in ("view", "str_replace"):
        # Same error PerDev's read_file/edit_file return (Node's fs message).
        name = "read_file" if cmd == "view" else "edit_file"
        new_args = {"path": path} if cmd == "view" else {
            "path": path, "old_str": args.get("old_str", ""), "new_str": args.get("new_str", "") or ""}
        return name, new_args, f"Error: ENOENT: no such file or directory, open '{path}'"
    if cmd == "view":
        return convert_view(args, obs)
    if cmd == "create":
        content = rel_text(args.get("file_text", ""))
        if "File created successfully" not in obs:
            raise Skip("create failed")
        return "write_file", {"path": path, "content": content}, f"Wrote {path} ({content.count(chr(10)) + 1} lines)."
    if cmd == "str_replace":
        new_args = {"path": path, "old_str": rel_text(args.get("old_str", "")), "new_str": rel_text(args.get("new_str", "") or "")}
        if "has been edited" in obs:
            return "edit_file", new_args, f"Edited {path}."
        if "did not appear verbatim" in obs:
            return "edit_file", new_args, (
                "Error: old_str was not found in the file. Read the file again and copy the text exactly, including indentation."
            )
        m = re.search(r"Multiple occurrences of old_str.*?lines \[([^\]]*)\]", obs, re.S)
        if m:
            n = len([x for x in m.group(1).split(",") if x.strip()])
            return "edit_file", new_args, f"Error: old_str occurs {n} times. Include more surrounding lines so it is unique."
        raise Skip("unknown str_replace result")
    raise Skip(f"unsupported editor command: {cmd}")


def parse_args(raw) -> dict:
    if isinstance(raw, dict):
        return raw
    try:
        return json.loads(raw or "{}")
    except json.JSONDecodeError as e:
        raise Skip("bad tool arguments") from e


def convert_row(row: dict, max_chars: int = 40000) -> dict:
    if not row.get("resolved"):
        raise Skip("not resolved")
    msgs = row["messages"]
    if isinstance(msgs, str):
        msgs = json.loads(msgs)
    repo = re.sub(r"[^A-Za-z0-9_.-]", "-", (row.get("instance_id") or "project").split(".")[0])

    out = [{"role": "system", "content": system_prompt(repo)}]
    # Tool results arrive after the assistant message that requested them.
    pending: dict[str, tuple[str, dict]] = {}
    finished = False

    for m in msgs:
        role = m.get("role")
        if role == "system":
            continue
        if role == "user":
            if len(out) > 1:
                raise Skip("multi-turn user messages")
            out.append({"role": "user", "content": clean_user(text_of(m.get("content")))})
            continue
        if role == "assistant":
            content = rel_text(text_of(m.get("content"))).strip()
            calls = m.get("tool_calls") or []
            if any(c["function"]["name"] == "submit" for c in calls):
                if len(content) < 40:
                    raise Skip("no final summary")
                out.append({"role": "assistant", "content": content})
                finished = True
                break
            new_calls = []
            for c in calls:
                cid = c.get("id") or f"call_{len(out)}_{len(new_calls)}"
                pending[cid] = (c["function"]["name"], parse_args(c["function"].get("arguments")))
                new_calls.append(cid)
            msg = {"role": "assistant", "content": content or None}
            if new_calls:
                msg["tool_calls"] = new_calls  # filled in when results arrive
            out.append(msg)
            continue
        if role == "tool":
            ids = m.get("tool_call_ids") or [m.get("tool_call_id")]
            obs = observation(m.get("content"))
            for cid in ids:
                if cid not in pending:
                    raise Skip("tool result without call")
                name, args = pending.pop(cid)
                if name == "bash":
                    command = rel_text(args.get("command", ""))
                    if not command.strip():
                        raise Skip("empty command")
                    if obs.startswith("Your command ran successfully and did not produce any output"):
                        result = "exit code: 0\n" + FORMAT["no_output_note"]
                    else:
                        result = rel_text(obs)
                    new_name, new_args = "run_command", {"command": command}
                elif name == "str_replace_editor":
                    new_name, new_args, result = convert_editor(args, obs)
                else:
                    raise Skip(f"unknown tool {name}")
                # Replace the placeholder id in the matching assistant message.
                for prev in reversed(out):
                    if prev["role"] == "assistant" and cid in (prev.get("tool_calls") or []):
                        i = prev["tool_calls"].index(cid)
                        prev["tool_calls"][i] = {
                            "id": cid,
                            "type": "function",
                            "function": {"name": new_name, "arguments": json.dumps(new_args, ensure_ascii=False)},
                        }
                        break
                out.append({"role": "tool", "tool_call_id": cid, "name": new_name, "content": truncate(result)})
            continue

    if not finished:
        raise Skip("never finished")
    if pending:
        raise Skip("unanswered tool calls")
    for m in out:
        if m["role"] == "assistant" and any(isinstance(c, str) for c in m.get("tool_calls", [])):
            raise Skip("unanswered tool calls")
    # Source outputs mix \r\n and \n; PerDev's edit_file works in \n, so normalize everything.
    for m in out:
        if m.get("content"):
            m["content"] = m["content"].replace("\r\n", "\n")
        for c in m.get("tool_calls") or []:
            args = {k: v.replace("\r\n", "\n") if isinstance(v, str) else v for k, v in json.loads(c["function"]["arguments"]).items()}
            c["function"]["arguments"] = json.dumps(args, ensure_ascii=False)
    size = sum(len(m.get("content") or "") + len(json.dumps(m.get("tool_calls", ""))) for m in out)
    if size > max_chars:
        raise Skip("too long")
    return {"messages": out, "source": "swe-smith", "id": row.get("traj_id") or row.get("instance_id")}


def iter_rows(path: Path):
    text = path.read_text(encoding="utf-8")
    if path.suffix == ".jsonl":
        for line in text.splitlines():
            if line.strip():
                yield json.loads(line)
        return
    data = json.loads(text)
    rows = data.get("rows", data) if isinstance(data, dict) else data
    for r in rows:
        yield r.get("row", r)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--input", required=True, type=Path)
    ap.add_argument("--out", required=True, type=Path)
    ap.add_argument("--max-chars", type=int, default=40000, help="skip conversations longer than this")
    a = ap.parse_args()

    kept, skipped = 0, {}
    with a.out.open("w", encoding="utf-8") as f:
        for row in iter_rows(a.input):
            try:
                ex = convert_row(row, a.max_chars)
            except Skip as e:
                reason = str(e).split(":")[0]
                skipped[reason] = skipped.get(reason, 0) + 1
                continue
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
            kept += 1
    print(f"kept {kept}, skipped {sum(skipped.values())}: {skipped}")


if __name__ == "__main__":
    main()

## Settings

In [ ]:
BASE_MODEL = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"  # 4-bit copy of Qwen/Qwen2.5-Coder-7B-Instruct
BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"           # used for the GGUF adapter's metadata
MAX_SEQ = 16384     # matches PerDev's runtime context (perdev.server.chatContext)
N_TRAIN = 800       # training conversations; the time guard below stops early if needed
N_EVAL = 30         # held-out conversations for the before/after score
LORA_R = 16
LEARNING_RATE = 1e-4
MAX_HOURS = 9.0     # stop training in time to export inside Kaggle's 12 h limit
SEED = 0

## 1. Build the dataset

In [ ]:
import glob, json, random
from datasets import load_dataset
import build_dataset as bd

random.seed(SEED)
raw = load_dataset("SWE-bench/SWE-smith-trajectories", split="tool")
raw = raw.filter(lambda r: r["resolved"]).shuffle(seed=SEED)

examples, skipped = [], {}
for row in raw:
    try:
        examples.append(bd.convert_row(row, max_chars=10**9))  # token length is checked later
    except bd.Skip as e:
        k = str(e).split(":")[0]
        skipped[k] = skipped.get(k, 0) + 1
    if len(examples) >= 3 * (N_TRAIN + N_EVAL):
        break
print(f"converted {len(examples)} SWE-smith conversations; skipped {skipped}")

own = []
for f in glob.glob("/kaggle/input/**/chat.jsonl", recursive=True):
    own += [json.loads(l) for l in open(f, encoding="utf-8") if l.strip()]
print(f"your own PerDev sessions: {len(own)}")

## 2. Load the model and render conversations with Qwen's chat template

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True)
TOOLS = json.load(open("perdev_format.json"))["tools"]
TOOL_NAMES = {t["function"]["name"]: t["function"]["parameters"].get("required", []) for t in TOOLS}

def to_hf(messages):
    # The chat template wants tool-call arguments as dicts, not JSON strings.
    out = []
    for m in messages:
        m = dict(m)
        if m.get("tool_calls"):
            m["tool_calls"] = [
                {"type": "function", "function": {"name": c["function"]["name"],
                 "arguments": json.loads(c["function"]["arguments"]) if isinstance(c["function"]["arguments"], str) else c["function"]["arguments"]}}
                for c in m["tool_calls"]
            ]
        if m.get("content") is None:
            m["content"] = ""
        out.append(m)
    return out

def render(messages, add_generation_prompt=False):
    return tokenizer.apply_chat_template(to_hf(messages), tools=TOOLS, tokenize=False, add_generation_prompt=add_generation_prompt)

def n_tokens(text):
    return len(tokenizer(text, add_special_tokens=False).input_ids)

fits = [e for e in examples if n_tokens(render(e["messages"])) <= MAX_SEQ]
print(f"{len(fits)} of {len(examples)} SWE-smith conversations fit in {MAX_SEQ} tokens")
random.shuffle(fits)
eval_set = fits[:N_EVAL]
train_set = fits[N_EVAL:N_EVAL + N_TRAIN] + [e for e in own if n_tokens(render(e["messages"])) <= MAX_SEQ] * 3  # upweight your own data
random.shuffle(train_set)
print(f"train: {len(train_set)}  eval: {len(eval_set)}")

## 3. Score the base model first

For each held-out conversation, cut it just before a random assistant step and let the model produce that step.
A step counts as **valid** when it is either a well-formed call to a real PerDev tool with its required arguments,
or (when the reference step is the final answer) a plain answer with no tool call.
**Same tool** means it chose the same tool the successful reference agent used.

In [ ]:
import re

def score(model, eval_set, max_new_tokens=384):
    FastLanguageModel.for_inference(model)
    rng = random.Random(SEED)
    valid = same = 0
    for ex in eval_set:
        msgs = ex["messages"]
        steps = [i for i, m in enumerate(msgs) if m["role"] == "assistant" and i > 1]
        i = rng.choice(steps)
        prompt = render(msgs[:i], add_generation_prompt=True)
        ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
        if ids.input_ids.shape[1] > MAX_SEQ - max_new_tokens:
            ids = tokenizer(render(msgs[:2], add_generation_prompt=True), return_tensors="pt", add_special_tokens=False).to("cuda")
            i = 2
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
        text = tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=False)
        calls = re.findall(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", text, re.S)
        ref_calls = msgs[i].get("tool_calls") or []
        ok = False
        if not ref_calls:
            ok = not calls and "<tool_call>" not in text
        elif calls:
            try:
                c = json.loads(calls[0])
                args = c.get("arguments", {})
                ok = c.get("name") in TOOL_NAMES and all(k in args for k in TOOL_NAMES[c["name"]])
                same += ok and c["name"] == ref_calls[0]["function"]["name"]
            except json.JSONDecodeError:
                pass
        valid += ok
    n = len(eval_set)
    return {"valid_step": round(valid / n, 3), "same_tool_as_reference": round(same / n, 3)}

before = score(model, eval_set)
print("base model:", before)

## 4. Train the LoRA adapter (only the assistant's turns are learned)

In [ ]:
import inspect, time
from datasets import Dataset
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED,
)
FastLanguageModel.for_training(model)

class TimeGuard(TrainerCallback):
    def __init__(self, hours):
        self.deadline = time.time() + hours * 3600
    def on_step_end(self, args, state, control, **kw):
        if time.time() > self.deadline:
            print(f"Time limit reached at step {state.global_step}; stopping and keeping what was learned.")
            control.should_training_stop = True

fields = SFTConfig.__dataclass_fields__
cfg = dict(
    dataset_text_field="text", per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=1, learning_rate=LEARNING_RATE, warmup_ratio=0.03, lr_scheduler_type="cosine",
    logging_steps=5, optim="adamw_8bit", fp16=True, bf16=False, output_dir="outputs",
    save_strategy="no", report_to="none", seed=SEED,
)
cfg["max_length" if "max_length" in fields else "max_seq_length"] = MAX_SEQ
tok_arg = "processing_class" if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters else "tokenizer"

trainer = SFTTrainer(
    model=model, train_dataset=Dataset.from_list([{"text": render(e["messages"])} for e in train_set]),
    args=SFTConfig(**cfg), callbacks=[TimeGuard(MAX_HOURS)], **{tok_arg: tokenizer},
)
# Tool results are rendered inside user turns by Qwen's template, so they are masked too.
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(stats)

## 5. Score again

In [ ]:
after = score(model, eval_set)
print("base model:      ", before)
print("with PerDev LoRA:", after)

## 6. Export the adapter for llama.cpp

In [ ]:
model.save_pretrained("perdev-lora")
tokenizer.save_pretrained("perdev-lora")
json.dump({"before": before, "after": after, "train_examples": len(train_set), "base": BASE_MODEL_ID},
          open("perdev-lora/eval.json", "w"), indent=2)

!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
!pip install -q /tmp/llama.cpp/gguf-py
!python /tmp/llama.cpp/convert_lora_to_gguf.py perdev-lora --base-model-id {BASE_MODEL_ID} --outtype f16 --outfile perdev-lora-f16.gguf
!rm -rf outputs
!ls -la /kaggle/working

## Use it in PerDev

1. Download `perdev-lora-f16.gguf` from this version's **Output** tab and save it to `E:\Tools\models\`.
2. In VS Code settings set **PerDev › Server: Chat Lora Path** to `E:\Tools\models\perdev-lora-f16.gguf`.
3. Restart VS Code (or run *PerDev: Start Local Model Servers*). The chat server now loads the 7B model plus your adapter.

Only keep the adapter if `with PerDev LoRA` scores better than `base model` above.